# 🐼 Pandas 3‑Hour Workout

*Generated on 2025-09-01 15:01*

This notebook is designed as a hands‑on sprint to level up your pandas skills in ~3 hours.

**Structure:**
1) Warmup (Titanic) — 30 min
2) Time Series (Stocks via `yfinance`) — 60 min
3) Clean & Analyze (Flights dataset) — 60 min
4) Mini‑Project (Merging, returns, correlation) — 30 min

> Tip: Work straight through, then revisit any starred challenges at the end.


## 0. Setup
- Install/Import dependencies
- Configure pandas display


In [ ]:
# If needed, install yfinance (uncomment next line)
# !pip install yfinance --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional but nice
import seaborn as sns

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


## 1) Warmup (Titanic) — 30 min

**Goals:**
- Quick EDA (`head`, `info`, `describe`)
- Filtering & boolean indexing
- GroupBy & pivot
- Feature engineering (family size)

**Dataset:** `seaborn.load_dataset("titanic")`


In [ ]:
# Load Titanic from seaborn
titanic = sns.load_dataset("titanic")
titanic.head()


In [ ]:
# Inspect
print(titanic.shape)
titanic.info()
titanic.describe(include="all").T.head(20)


### Exercises
1. Count missing values per column.
2. Fill `age` with the median and drop rows with missing `embarked`.
3. Create `family_size = sibsp + parch + 1` and `is_child = age < 12`.
4. Find survival rate by `sex` and by `class`.
5. Build a pivot table of **mean fare** by `sex` (rows) and `class` (columns).
6. **Advanced:** Compute survival rate for `who` (child/woman/man) and sort descending.


In [ ]:
# 1) Missing values
titanic.isnull().sum()


In [ ]:
# 2) Fill/drop
titanic["age"] = titanic["age"].fillna(titanic["age"].median())
titanic = titanic.dropna(subset=["embarked"])
titanic.isnull().sum().head(10)


In [ ]:
# 3) New columns
titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1
titanic["is_child"] = titanic["age"] < 12
titanic[["sibsp","parch","family_size","is_child"]].head()


In [ ]:
# 4) Survival rates
surv_by_sex = titanic.groupby("sex")["survived"].mean().sort_values(ascending=False)
surv_by_class = titanic.groupby("class")["survived"].mean().sort_values(ascending=False)
surv_by_sex, surv_by_class


In [ ]:
# 5) Pivot table
titanic.pivot_table(values="fare", index="sex", columns="class", aggfunc="mean").round(2)


In [ ]:
# 6) Advanced: survival by 'who'
survival_rate = pd.DataFrame()
survival_rate["total"] = titanic["who"].value_counts()
survival_rate["survived"] = titanic[titanic["alive"]=="yes"]["who"].value_counts()
survival_rate["survival_rate"] = survival_rate["survived"] / survival_rate["total"]
survival_rate.sort_values("survival_rate", ascending=False)


### Bonus: Does family size affect survival?
Create a plot of survival rate vs `family_size`.


In [ ]:
fam = (titanic.groupby("family_size")["survived"].mean().reset_index())
plt.figure()
plt.plot(fam["family_size"], fam["survived"], marker="o")
plt.title("Survival Rate vs Family Size")
plt.xlabel("Family Size")
plt.ylabel("Survival Rate")
plt.show()


## 2) Time Series (Stocks via `yfinance`) — 60 min

**Goals:**
- Download multi-ticker OHLCV
- Daily % returns, rolling means
- Resampling to monthly
- Compare performance across tickers

> If `yfinance` import fails, run the `pip install` cell at the top.


In [ ]:
import yfinance as yf

tickers = "AAPL MSFT TSLA"
prices = yf.download(tickers, start="2020-01-01", end="2024-12-31", auto_adjust=True, progress=False)
prices.head()


In [ ]:
# Inspect structure (MultiIndex columns: (Field, Ticker))
prices.columns


### Exercises
1. Create a **Close-only** DataFrame with columns `AAPL`, `MSFT`, `TSLA`.
2. Compute **daily returns** with `pct_change()`.
3. Plot a **30‑day moving average** for AAPL close.
4. Resample to **monthly returns** (end of month).
5. Which ticker had the **best return** in calendar year 2022?


In [ ]:
# 1) Close-only wide table
close_wide = prices["Close"].copy()
close_wide.head()


In [ ]:
# 2) Daily returns
daily_ret = close_wide.pct_change().dropna()
daily_ret.head()


In [ ]:
# 3) 30-day moving average (AAPL)
ma30 = close_wide["AAPL"].rolling(window=30).mean()
plt.figure()
plt.plot(close_wide.index, close_wide["AAPL"], label="AAPL Close")
plt.plot(ma30.index, ma30, label="AAPL 30D MA")
plt.legend()
plt.title("AAPL Close vs 30-Day MA")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()


In [ ]:
# 4) Monthly returns (end of month)
monthly_price = close_wide.resample("M").last()
monthly_ret = monthly_price.pct_change().dropna()
monthly_ret.head()


In [ ]:
# 5) Best performer in 2022
yr = 2022
r_2022 = (close_wide[close_wide.index.year == yr]
          .pipe(lambda df: df.iloc[-1] / df.iloc[0] - 1))
r_2022.sort_values(ascending=False)


## 3) Clean & Analyze (Flights) — 60 min

**Goals:**
- Convert year/month to datetime
- Pivot into a heatmap table
- Rolling averages
- Simple insights

We'll use seaborn's built-in `flights` dataset.


In [ ]:
flights = sns.load_dataset("flights")
flights.head()


In [ ]:
# Make a proper datetime column
flights["date"] = pd.to_datetime(flights["year"].astype(str) + "-" + flights["month"].astype(str) + "-01")
flights.set_index("date", inplace=True)
flights.sort_index(inplace=True)
flights.head()


### Exercises
1. Pivot passengers by `year` (rows) and `month` (cols).
2. Compute a 12‑month rolling average of `passengers`.
3. Find the busiest year by total passengers.


In [ ]:
# 1) Pivot table
pivot = flights.pivot_table(values="passengers",
                            index="year",
                            columns="month",
                            aggfunc="sum")
pivot


In [ ]:
# 2) Rolling average (12 months)
roll12 = flights["passengers"].rolling(12).mean()
plt.figure()
plt.plot(flights.index, flights["passengers"], label="Passengers")
plt.plot(roll12.index, roll12, label="12M Rolling Avg")
plt.legend()
plt.title("Passengers with 12‑Month Rolling Average")
plt.xlabel("Date")
plt.ylabel("Passengers")
plt.show()


In [ ]:
# 3) Busiest year
busiest = flights.groupby(flights.index.year)["passengers"].sum().sort_values(ascending=False)
busiest.head()


## 4) Mini‑Project (30 min): Merge, Analyze, Visualize

**Objective:**
- Combine multiple tickers, compute returns, and analyze correlation.

**Tasks:**
1. Using `close_wide` from Section 2, compute **monthly returns** for AAPL, MSFT, TSLA.
2. Build a **correlation matrix** of monthly returns.
3. Plot the correlation matrix as a heatmap.
4. (Bonus) Identify the pair with **highest correlation** and the pair with **lowest correlation**.


In [ ]:
# 1) Monthly returns (reusing monthly_ret from Section 2 if available)
if 'monthly_ret' not in globals():
    monthly_price = close_wide.resample("M").last()
    monthly_ret = monthly_price.pct_change().dropna()

monthly_ret.head()


In [ ]:
# 2) Correlation matrix
corr = monthly_ret.corr()
corr


In [ ]:
# 3) Heatmap (matplotlib only, no seaborn styles enforced)
plt.figure()
plt.imshow(corr, interpolation="nearest")
plt.title("Monthly Return Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:
# 4) Bonus: most/least correlated pairs
pairs = []
cols = corr.columns.tolist()
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        pairs.append(((cols[i], cols[j]), corr.iloc[i, j]))
sorted_pairs = sorted(pairs, key=lambda x: x[1], reverse=True)
most_corr = sorted_pairs[0]
least_corr = sorted_pairs[-1]
most_corr, least_corr
